# AIAvatarLearningApp - Train the Stage 2 feedback model in Colab

Two paths through this notebook:

- **Recommended (local demo)**: cells 1-7. Trains the model, converts it to a quantized GGUF (~2 GB), saves to Drive. Then download to your laptop and run via Ollama. Fully offline demo.
- **Alternative (Colab-hosted)**: cells 1-5, then 8. Serves the model from Colab via ngrok. Requires internet + an active Colab session during your demo.

**Before you start:** Runtime -> Change runtime type -> GPU -> A100 (or T4 if A100 isn't available). Save this notebook to Drive so it survives disconnects.

## 1. Confirm a GPU is attached

In [ ]:
!nvidia-smi

## 2. Clone the repo

If the repo is private, swap the line for:
`!git clone https://<YOUR_GITHUB_TOKEN>@github.com/YuvalHaski/AIAvatarLearningApp.git`

In [ ]:
!git clone https://github.com/YuvalHaski/AIAvatarLearningApp.git
%cd AIAvatarLearningApp

## 3. Install training dependencies (~3 min)

In [ ]:
!pip install -q -r training/requirements.txt

## 4. Mount Google Drive for persistent output

Critical - Colab sessions disconnect, but anything written to Drive survives. The trained model (~7 GB) and the final GGUF (~2 GB) will both be saved there.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/aiavatar_model

## 5. Train (~30-90 min depending on GPU)

Watch the eval loss decrease across the 3 epochs. Final output: `/content/drive/MyDrive/aiavatar_model/merged/`. Keep this browser tab visible while it runs.

In [ ]:
!python training/train.py \
    --data training/data \
    --out /content/drive/MyDrive/aiavatar_model

## 6. Convert the trained model to a quantized GGUF

Builds llama.cpp's tools, converts `merged/` -> F16 GGUF, then quantizes to Q4_K_M (~2.5 GB). The quantized file is what you'll download to your laptop and run via Ollama.

Takes ~5-10 minutes total.

In [ ]:
# Build llama.cpp's quantize tool from source. ~3 min.
!git clone --depth=1 https://github.com/ggerganov/llama.cpp.git /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements.txt
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DLLAMA_CURL=OFF -DGGML_CUDA=OFF
!cmake --build /content/llama.cpp/build --config Release -j --target llama-quantize

In [ ]:
# Convert the merged HF model to F16 GGUF (~7 GB, scratch space only).
!python /content/llama.cpp/convert_hf_to_gguf.py \
    /content/drive/MyDrive/aiavatar_model/merged \
    --outfile /content/aiavatar-feedback-f16.gguf \
    --outtype f16

In [ ]:
# Quantize to Q4_K_M and save directly to Drive (~2.5 GB).
!/content/llama.cpp/build/bin/llama-quantize \
    /content/aiavatar-feedback-f16.gguf \
    /content/drive/MyDrive/aiavatar_model/aiavatar-feedback-q4_k_m.gguf \
    Q4_K_M

In [ ]:
# Verify the GGUF landed in Drive.
!ls -lh /content/drive/MyDrive/aiavatar_model/*.gguf

## 7. Download the GGUF to your laptop and serve via Ollama

Do these steps on your **Windows laptop**, not in Colab:

1. Open Google Drive in a browser, navigate to `MyDrive/aiavatar_model/`, and download `aiavatar-feedback-q4_k_m.gguf` (~2.5 GB).
2. Move it to your project, e.g. `C:\Users\User\FinalProjectAIAvatar\AIAvatarLearningApp\models\aiavatar-feedback-q4_k_m.gguf`.
3. Install Ollama for Windows from https://ollama.com/download.
4. In the same folder as the `.gguf`, create a text file named `Modelfile` (no extension) with this content:
   ```
   FROM ./aiavatar-feedback-q4_k_m.gguf
   PARAMETER temperature 0.3
   ```
5. In PowerShell, from that folder:
   ```powershell
   ollama create feedback-model -f Modelfile
   ollama serve   # if not already running as a service
   ```
6. In your `.env`, set:
   ```
   FEEDBACK_MODEL_URL=http://localhost:11434/v1
   FEEDBACK_MODEL_NAME=feedback-model
   ```
7. Restart your FastAPI backend. Feedback now comes from your fine-tuned model running fully on your laptop.

If you have an NVIDIA GPU, Ollama uses it automatically. Otherwise it runs on CPU - usable but slower (~3-8s per response).

## 8. (Alternative) Serve from Colab via ngrok

Only use this if you can't run Ollama locally for some reason. Tied to the Colab session - your laptop must have internet during the demo, and Colab can disconnect.

Sign up for a free ngrok account at https://dashboard.ngrok.com/signup, copy your auth token, and paste it below.

In [ ]:
!pip install -q vllm pyngrok

NGROK_TOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"
!ngrok config add-authtoken {NGROK_TOKEN}

In [ ]:
import subprocess, time

proc = subprocess.Popen([
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", "/content/drive/MyDrive/aiavatar_model/merged",
    "--served-model-name", "feedback-model",
    "--port", "8000",
])

print("waiting 60s for vLLM to come up...")
time.sleep(60)

from pyngrok import ngrok
tunnel = ngrok.connect(8000)
print()
print("Put this in your local .env:")
print(f"  FEEDBACK_MODEL_URL={tunnel.public_url}/v1")

## 9. (Optional) Evaluate the trained model on the val set

Runs the faithfulness checks from `training/evaluate.py` against the running vLLM server. Only works if you ran section 8 first (Ollama runs on your laptop, not in Colab).

In [ ]:
!python training/evaluate.py \
    --endpoint http://localhost:8000/v1 \
    --model-name feedback-model \
    --val training/data/val.jsonl